In [ ]:
# -*- coding: utf-8 -*-
"""
DrumWavToMidi.ipynb
WAVs de batería → MIDIs → Tokens para entrenar
"""

# !pip install -q basic-pitch pretty-midi tqdm

import os, re, pickle
from pathlib import Path
import pretty_midi
from tqdm import tqdm
from basic_pitch.inference import predict
from basic_pitch import ICASSP_2022_MODEL_PATH

from google.colab import drive
drive.mount('/content/drive')

base        = '/content/drive/MyDrive/Musiko/LoFiModel'
carpeta_wav = f'{base}/Data/drums_wav'
carpeta_mid = f'{base}/Data/drums_midi'
os.makedirs(carpeta_mid, exist_ok=True)

# ── Mapa de snap: pitch detectado → pitch GM estándar ──────────
def snap_gm(pitch):
    if pitch < 40:  return 36  # Kick
    if pitch < 55:  return 38  # Snare
    if pitch < 66:  return 42  # HH Closed
    if pitch < 75:  return 46  # HH Open
    return 51                  # Ride / otros

# ── Paso 1: WAV → MIDI con Basic Pitch ─────────────────────────
wavs = list(Path(carpeta_wav).rglob("*.wav"))
print(f"Transcribiendo {len(wavs)} WAVs...")

for wav in tqdm(wavs):
    salida = Path(carpeta_mid) / (wav.stem + '.mid')
    if salida.exists(): continue
    try:
        _, midi, _ = predict(str(wav), ICASSP_2022_MODEL_PATH,
                             onset_threshold=0.5, frame_threshold=0.3,
                             minimum_note_length=50, melodia_trick=False)
        midi.write(str(salida))
    except Exception as e:
        print(f"  Error: {wav.name} — {e}")

# ── Paso 2: Limpiar MIDIs + snap a GM ──────────────────────────
def limpiar(midi_path):
    bpm  = int(re.search(r'(\d+)bpm', midi_path.stem).group(1)) if re.search(r'(\d+)bpm', midi_path.stem) else 85
    midi = pretty_midi.PrettyMIDI(initial_tempo=bpm)
    inst = pretty_midi.Instrument(program=0, is_drum=True)
    s16  = (60 / bpm) / 4  # duración de semicorchea

    for i in pretty_midi.PrettyMIDI(str(midi_path)).instruments:
        for n in i.notes:
            if n.end - n.start < 0.02: continue  # filtra artefactos
            t = round(n.start / s16) * s16        # cuantiza al 16avo
            inst.notes.append(pretty_midi.Note(n.velocity, snap_gm(n.pitch), t, t + s16 * 0.8))

    if len(inst.notes) < 16: return None
    midi.instruments.append(inst)
    return midi

print("\nLimpiando MIDIs...")
validos = 0
for mid in tqdm(list(Path(carpeta_mid).glob("*.mid"))):
    limpio = limpiar(mid)
    if limpio:
        limpio.write(str(mid))  # sobreescribe con la versión limpia
        validos += 1

print(f"MIDIs válidos: {validos}")

# ── Paso 3: MIDI → Tokens ──────────────────────────────────────
def a_tokens(midi_path):
    midi   = pretty_midi.PrettyMIDI(str(midi_path))
    eventos = sorted([
        {'t': n.start, 'tipo': 'on',  'p': n.pitch, 'v': n.velocity}
        for i in midi.instruments for n in i.notes
    ] + [
        {'t': n.end,   'tipo': 'off', 'p': n.pitch, 'v': n.velocity}
        for i in midi.instruments for n in i.notes
    ], key=lambda e: e['t'])

    tokens, t_actual = [], 0.0
    i = 0
    while i < len(eventos):
        delta = eventos[i]['t'] - t_actual
        if delta > 0.01:
            pasos = int(delta / 0.01)
            while pasos > 0:
                s = min(99, pasos); tokens.append(256 + s); pasos -= s
            t_actual = eventos[i]['t']

        g = []
        while i < len(eventos) and abs(eventos[i]['t'] - eventos[i-1 if i>0 else 0]['t']) < 0.005:
            g.append(eventos[i]); i += 1
        for e in g:
            tokens.append(356 + min(e['v'] // 4, 31))
            tokens.append(e['p'] if e['tipo'] == 'on' else e['p'] + 128)

    return tokens if len(tokens) > 50 else None

print("\nTokenizando...")
secuencias = [t for mid in tqdm(list(Path(carpeta_mid).glob("*.mid")))
              if (t := a_tokens(mid)) is not None]

with open(f'{base}/Data/drums_tokens.pkl', 'wb') as f:
    pickle.dump(secuencias, f)

print(f"\n✓ {len(secuencias)} secuencias guardadas en drums_tokens.pkl")